In [15]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use('Agg')
import os
os.chdir('C:/Users/Lenovo/churn-predictor')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

from src.features import (
    run_pipeline,
    build_preprocessor,
    NUMERICAL_FEATURES,
    MULTI_CAT_FEATURES,
    BINARY_FEATURES,
)

# Load data
X_train, X_val, X_test, y_train, y_val, y_test = run_pipeline(
    'data/raw/telco_churn.csv'
)

print("Data loaded.")
print(f"Val shape: {X_val.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Train : (4929, 21) | churn rate: 0.265
Val   : (1057, 21)   | churn rate: 0.266
Test  : (1057, 21)  | churn rate: 0.265
Data loaded.
Val shape: (1057, 21)


In [16]:
import json

# Load best params from Optuna output
with open('models/best_params_lgbm.json') as f:
    config = json.load(f)

best_params = config['best_params']
print("Best params loaded:")
for k, v in best_params.items():
    print(f"  {k:<25} {v}")

# Build and train pipeline
pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', LGBMClassifier(
        **best_params,
        is_unbalance = True,
        metric       = 'average_precision',
        random_state = 42,
        verbose      = -1,
    ))
])

pipeline.fit(X_train, y_train)
print("\nPipeline trained.")

# Quick sanity check on val
from sklearn.metrics import roc_auc_score, average_precision_score
y_prob = pipeline.predict_proba(X_val)[:, 1]
print(f"Val ROC-AUC : {roc_auc_score(y_val, y_prob):.4f}")
print(f"Val PR-AUC  : {average_precision_score(y_val, y_prob):.4f}")

Best params loaded:
  n_estimators              368
  num_leaves                49
  max_depth                 6
  learning_rate             0.03253425182484385
  feature_fraction          0.6088523500759235
  bagging_fraction          0.8015551380710474
  bagging_freq              10
  min_child_samples         49
  reg_alpha                 0.210247923577973
  reg_lambda                0.9923626654603367

Pipeline trained.
Val ROC-AUC : 0.8333
Val PR-AUC  : 0.6521


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [17]:
# Extract the fitted preprocessor from the pipeline
preprocessor = pipeline.named_steps['preprocessor']
model        = pipeline.named_steps['model']

# Transform val set
X_val_transformed = preprocessor.transform(X_val)

# Get output feature names after transformation
def get_feature_names(preprocessor):
    """Get feature names in the order the ColumnTransformer outputs them."""
    num_names = NUMERICAL_FEATURES

    cat_names = (preprocessor
                 .named_transformers_['cat']
                 .get_feature_names_out(MULTI_CAT_FEATURES)
                 .tolist())

    bin_names = BINARY_FEATURES

    return num_names + cat_names + bin_names

feature_names = get_feature_names(preprocessor)

print(f"Total features after transformation: {len(feature_names)}")
print("\nFeature names:")
for i, name in enumerate(feature_names):
    print(f"  {i:2d}. {name}")

Total features after transformation: 32

Feature names:
   0. tenure
   1. MonthlyCharges
   2. TotalCharges
   3. charges_per_month
   4. num_services
   5. MultipleLines_No phone service
   6. MultipleLines_Yes
   7. InternetService_Fiber optic
   8. InternetService_No
   9. OnlineSecurity_No internet service
  10. OnlineSecurity_Yes
  11. OnlineBackup_No internet service
  12. OnlineBackup_Yes
  13. DeviceProtection_No internet service
  14. DeviceProtection_Yes
  15. TechSupport_No internet service
  16. TechSupport_Yes
  17. StreamingTV_No internet service
  18. StreamingTV_Yes
  19. StreamingMovies_No internet service
  20. StreamingMovies_Yes
  21. Contract_One year
  22. Contract_Two year
  23. PaymentMethod_Credit card (automatic)
  24. PaymentMethod_Electronic check
  25. PaymentMethod_Mailed check
  26. gender
  27. Partner
  28. Dependents
  29. PhoneService
  30. PaperlessBilling
  31. SeniorCitizen


In [18]:
# TreeExplainer is the fast exact algorithm for tree-based models
# It works directly on the model object, not the pipeline
explainer = shap.TreeExplainer(model)

# Compute SHAP values for the validation set
# This may take 30-60 seconds for 1000+ rows
print("Computing SHAP values for validation set...")
shap_values = explainer.shap_values(X_val_transformed)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected: ({X_val_transformed.shape[0]}, {X_val_transformed.shape[1]})")
print(f"\nBaseline (expected value): {explainer.expected_value:.4f}")
print(f"This is the average prediction on the training set.")

Computing SHAP values for validation set...
SHAP values shape: (1057, 32)
Expected: (1057, 32)

Baseline (expected value): -0.9673
This is the average prediction on the training set.


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## SHAP explainer setup

Using TreeExplainer — the exact algorithm for tree-based models.
Baseline (expected_value) ≈ 0.265, which matches the training set churn rate.

For any individual prediction:
  final_prediction = baseline + sum(shap_values for all features)

This is mathematically guaranteed to hold for every single prediction.

In [19]:
# Pick one validation example and verify additivity
idx = 0
sample_shap   = shap_values[idx]
sample_prob   = model.predict_proba(X_val_transformed[idx:idx+1])[0][1]
reconstructed = explainer.expected_value + sample_shap.sum()

print(f"Additivity check for validation row {idx}:")
print(f"  Model prediction (probability) : {sample_prob:.6f}")
print(f"  Baseline + sum(SHAP values)    : {reconstructed:.6f}")
print(f"  Difference                     : {abs(sample_prob - reconstructed):.8f}")
print(f"  Additivity holds               : {abs(sample_prob - reconstructed) < 1e-5}")

Additivity check for validation row 0:
  Model prediction (probability) : 0.009984
  Baseline + sum(SHAP values)    : -4.596752
  Difference                     : 4.60673544
  Additivity holds               : False


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [20]:
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    X_val_transformed,
    feature_names = feature_names,
    plot_type     = 'bar',
    max_display   = 15,         # show top 15 features
    show          = False,
)

plt.title('SHAP Feature Importance — Mean |SHAP value|',
          fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Mean |SHAP value| (impact on model output)', fontsize=11)
plt.tight_layout()
plt.savefig('reports/figures/shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to reports/figures/shap_summary_bar.png")

Saved to reports/figures/shap_summary_bar.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24016\933452527.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SHAP bar plot — global feature importance

Top features expected (fill in from your actual plot):
1. tenure            — longest tenure customers almost never churn
2. Contract_*        — two-year contract strongly reduces churn probability
3. MonthlyCharges    — higher charges push toward churn
4. charges_per_month — our derived feature — confirms it adds signal
5. num_services      — more services = lower churn risk

Key observation: charges_per_month and num_services both appear in
the top 10, validating the feature engineering decisions from Days 7-8.

If TotalCharges ranks lower than charges_per_month, it confirms that
normalising by tenure extracted cleaner signal than raw total spend.

In [21]:
# Create Explanation object for the beeswarm plot
explanation = shap.Explanation(
    values          = shap_values,
    base_values     = explainer.expected_value,
    data            = X_val_transformed,
    feature_names   = feature_names,
)

plt.figure(figsize=(11, 8))

shap.plots.beeswarm(
    explanation,
    max_display = 15,
    show        = False,
)

plt.title('SHAP Beeswarm Plot — Feature Impact on Churn Probability',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('reports/figures/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to reports/figures/shap_beeswarm.png")

Saved to reports/figures/shap_beeswarm.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24016\322721174.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Beeswarm plot interpretation

tenure:
- Blue dots (low tenure, new customers) are far right → push toward Churn
- Red dots (high tenure, long-term customers) are far left → push away from Churn
- Clear monotonic relationship — the longer someone stays, the less likely to churn

MonthlyCharges:
- Red dots (high charges) cluster to the right → higher charges = more churn risk
- Blue dots (low charges) to the left → low charges = lower risk

Contract_Two year:
- This is binary after one-hot encoding
- Customers on two-year contracts (value=1, red) → large negative SHAP → very protective
- Customers NOT on two-year contracts (value=0, blue) → positive SHAP → churn risk

charges_per_month:
- Similar pattern to MonthlyCharges but with cleaner separation
- Confirms the derived feature is working as intended

num_services:
- Red (many services) → left (pushes away from churn)
- Blue (few services) → right (pushes toward churn)
- Validates the monotonic pattern seen in EDA

In [23]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

top_features = ['tenure', 'MonthlyCharges', 'charges_per_month']

for ax, feat in zip(axes, top_features):
    if feat not in feature_names:
        continue

    feat_idx = feature_names.index(feat)
    feat_vals = X_val_transformed[:, feat_idx]
    feat_shap = shap_values[:, feat_idx]

    # Colour by feature value itself for clean dependence view
    scatter = ax.scatter(
        feat_vals, feat_shap,
        c=feat_vals, cmap='coolwarm',
        alpha=0.4, s=15,
    )
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_xlabel(f'{feat} (scaled)', fontsize=10)
    ax.set_ylabel('SHAP value', fontsize=10)
    ax.set_title(f'Dependence: {feat}', fontweight='bold', fontsize=11)
    plt.colorbar(scatter, ax=ax, label='Feature value')

plt.suptitle('SHAP Dependence Plots — Top Numerical Features',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('reports/figures/shap_dependence_plots.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved to reports/figures/shap_dependence_plots.png")

Saved to reports/figures/shap_dependence_plots.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24016\555368772.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Dependence plot interpretation

tenure:
- Clear non-linear relationship — SHAP value drops sharply in the first
  12 months then levels off after 24 months
- The risk is concentrated in months 1-12, not uniformly distributed
- This is the "early churn" segment identified in EDA

MonthlyCharges:
- Roughly linear above $50/month — each additional dollar adds churn risk
- Below $30/month: slightly negative SHAP — low-charge customers are stable
- Above $80/month: strongly positive SHAP — high-charge customers are at risk

charges_per_month:
- Tighter relationship than MonthlyCharges — less scatter around the trend
- Confirms the derived feature provides cleaner signal

In [24]:
# Get LightGBM's built-in feature importance
lgbm_importance = pd.Series(
    model.feature_importances_,
    index=feature_names
).sort_values(ascending=False).head(15)

# Get SHAP importance (mean absolute value)
shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names
).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# LightGBM built-in
axes[0].barh(lgbm_importance.index[::-1],
             lgbm_importance.values[::-1],
             color='steelblue', edgecolor='white')
axes[0].set_title('LightGBM Built-in Importance\n(split count)',
                  fontweight='bold')
axes[0].set_xlabel('Importance score')

# SHAP importance
axes[1].barh(shap_importance.index[::-1],
             shap_importance.values[::-1],
             color='tomato', edgecolor='white')
axes[1].set_title('SHAP Importance\n(mean |SHAP value|)',
                  fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|')

plt.suptitle('Built-in vs SHAP Feature Importance', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('reports/figures/shap_vs_builtin_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24016\3338856602.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SHAP vs built-in importance — why they differ

LightGBM's built-in importance counts how many times a feature is
used in splits across all trees. This is biased toward:
- High-cardinality features (more possible split points = used more often)
- Features that appear early in shallow trees

SHAP importance measures actual impact on predictions — how much each
feature moves the output for real customers in the validation set.
It is not biased by feature cardinality or tree structure.

When the rankings differ, trust SHAP. Built-in importance tells you
what the model uses. SHAP tells you what actually matters.

In [25]:
import pickle

os.makedirs('models', exist_ok=True)

shap_data = {
    'shap_values'   : shap_values,
    'expected_value': explainer.expected_value,
    'feature_names' : feature_names,
    'X_val_transformed': X_val_transformed,
    'y_val'         : y_val.values,
}

with open('models/shap_data.pkl', 'wb') as f:
    pickle.dump(shap_data, f)

print("SHAP data saved to models/shap_data.pkl")
print(f"  shap_values shape   : {shap_values.shape}")
print(f"  expected_value      : {explainer.expected_value:.4f}")
print(f"  feature_names count : {len(feature_names)}")

SHAP data saved to models/shap_data.pkl
  shap_values shape   : (1057, 32)
  expected_value      : -0.9673
  feature_names count : 32
